In [28]:
import os
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# ==========================================
# 1. ENVIRONMENT CONFIGURATION
# ==========================================
load_dotenv(override=True)

# Update these strings if your .env file uses different variable names
api_key_string = os.environ.get("KEY")
model_env_string = os.environ.get("MODEL", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
base_url_string = os.environ.get("BASE_URL")

# Failsafe check for critical variables
if not api_key_string:
    raise ValueError("❌ 'ANTHROPIC_API_KEY' was not found in your environment or .env file.")

# 2. STATE GRAPH LAYOUT DEFINITION
# ==========================================
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

@tool
def grant_access(user: str, system: str) -> str:
    """Grants a specific user access to a targeted IT system."""
    return f"Access to '{system}' successfully GRANTED to user '{user}'."

@tool
def revoke_access(user: str, system: str) -> str:
    """Revokes a specific user's access from a targeted IT system."""
    return f"Access to '{system}' successfully REVOKED for user '{user}'."

tools = [grant_access, revoke_access]

# ==========================================
# 3. CUSTOM MODEL INITIALIZATION
# ==========================================
# Injecting the api_key, custom model name, and custom base URL gateway mapping
model = ChatAnthropic(
    model=model_env_string, 
    api_key=api_key_string,
    base_url=base_url_string
).bind_tools(tools)

def call_model(state: AgentState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "continue"
    return "end"

# ==========================================
# 4. GRAPH ASSEMBLY & FRESH COMPILATION
# ==========================================
workflow = StateGraph(AgentState)
workflow.add_node("agent", call_model)
workflow.add_node("action", ToolNode(tools))

workflow.add_edge(START, "agent")
workflow.add_conditional_edges(
    "agent", 
    should_continue, 
    {"continue": "action", "end": END}
)
workflow.add_edge("action", "agent")

memory = MemorySaver()
app = workflow.compile(checkpointer=memory, interrupt_before=["action"])

# ==========================================
# 5. STREAM EXECUTION & HITL CONTROL
# ==========================================
config = {"configurable": {"thread_id": "gateway_hitl_run"}}
initial_input = {"messages": [HumanMessage(content="Give database admin access to user 'Kiran'")]}

print("\n--- Starting Gateway Agent Execution ---")
for event in app.stream(initial_input, config, stream_mode="values"):
    last_message = event["messages"][-1]
    if hasattr(last_message, "content") and last_message.content:
        print(f"\n[Agent]: {last_message.content}")

snapshot = app.get_state(config)

if snapshot.next:
    print("\n" + "="*60)
    print("🚨 SYSTEM PAUSED: Human-in-the-Loop Authorization Required")
    pending_msg = snapshot.values["messages"][-1]
    for action_intent in pending_msg.tool_calls:
        print(f"👉 Requested Action: {action_intent.get('name', 'Unknown')}")
        print(f"👉 Target Context   : {action_intent.get('args', {})}")
    print("="*60 + "\n")
    
    manager_decision = input("Approve? (yes/no): ").strip().lower()
    
    if manager_decision == "yes":
        print("\n✅ Request Approved. Resuming processing workflow...")
        for event in app.stream(None, config, stream_mode="values"):
            last_message = event["messages"][-1]
            if hasattr(last_message, "content") and last_message.content:
                print(f"\n[Agent]: {last_message.content}")
    else:
        print("\n❌ Request Denied. Terminating processing thread.")
else:
    print("\nWorkflow processed to completion without operational interruptions.")


--- Starting Gateway Agent Execution ---

[Agent]: Give database admin access to user 'Kiran'

[Agent]: [{'id': 'toolu_bdrk_0111LBviDSaik2apFngM72Ao', 'input': {'user': 'Kiran', 'system': 'database admin'}, 'name': 'grant_access', 'type': 'tool_use'}]

🚨 SYSTEM PAUSED: Human-in-the-Loop Authorization Required
👉 Requested Action: grant_access
👉 Target Context   : {'user': 'Kiran', 'system': 'database admin'}


✅ Request Approved. Resuming processing workflow...

[Agent]: [{'id': 'toolu_bdrk_0111LBviDSaik2apFngM72Ao', 'input': {'user': 'Kiran', 'system': 'database admin'}, 'name': 'grant_access', 'type': 'tool_use'}]

[Agent]: Access to 'database admin' successfully GRANTED to user 'Kiran'.

[Agent]: Done! I've successfully granted database admin access to user 'Kiran'.
